# VLM-style fusion: log-sum-exp pooled layers + question + answer text

LLaVA-style hallucination classifier. The 29 saved Qwen2.5-7B answer-token states are log-sum-exp pooled into 15 layer windows (same pooling as the 4 × 128 MLP run). Each pooled vector is standardized with source-train statistics and mapped by a 2-layer MLP projector into Qwen2.5-0.5B-Instruct's embedding space as a soft token, like an image patch. The 15 soft tokens are followed by the question and the generated answer text; the LM is fine-tuned end to end and a linear head on the last token predicts BLEURT-correct vs hallucinated.

Same saved six-source split, BCE loss, seed, best-source-validation-AUROC checkpoint selection, threshold calibration and held-out TriviaQA evaluation as the MLP runs. Training: 3 epochs, batch 32, AdamW (LM 2e-5, projector/head 1e-4), warmup + cosine, mixed precision.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 02035cb
!pip install -q -r requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_vlm_fusion_logsumexp')
command = [sys.executable, '/kaggle/working/SALA/sala_vlm_fusion.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--seed', '42', '--train-device', 'cuda:0', '--epochs', '3', '--batch-size', '32']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Fusion LM:', summary['fusion_lm'], 'pooled layer tokens:', summary['pooled_layers'], 'best epoch:', summary['best_epoch'])
print('Source val AUROC:', summary['best_source_validation_auroc_during_training'])
print('Source test:', json.dumps(summary['source_test_metrics'], indent=2))
print('Target (TriviaQA):', json.dumps(summary['target_metrics'], indent=2))
display(results.head())